<a href="https://colab.research.google.com/github/yiruchen4/Chen2026/blob/main/SocialDoorInstructions/doordata_process.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

This handles:

*   Single files
*   Multiple files
*   Folder uploads (users zip the folder first)



In [ ]:
# @title This code cleans up door_outputs folder. If you don't want it, comment it or unrun it.
import os
import shutil

# ---- Empty door_outputs safely ----
OUTPUT_ROOT = "/content/door_outputs"

# Create if missing
os.makedirs(OUTPUT_ROOT, exist_ok=True)

# Remove everything inside
for item in os.listdir(OUTPUT_ROOT):
    path = os.path.join(OUTPUT_ROOT, item)
    if os.path.isfile(path) or os.path.islink(path):
        os.remove(path)
    elif os.path.isdir(path):
        shutil.rmtree(path)

print("door_outputs directory cleared.")

In [ ]:
# @title Upload all the DOOR data files (single, multiple, or a zipped folder). This code will identify the files with door events.
from google.colab import files
import os, glob, shutil, zipfile, re
import pandas as pd
import numpy as np

# ---- 1) Set up a persistent working directory in Colab ----
UPLOAD_STAGING = "/content/upload_staging"   # where uploads land / unzip happens
WORKDIR = "/content/working_files"           # where you keep "working" copies for pd.read_csv

os.makedirs(UPLOAD_STAGING, exist_ok=True)
os.makedirs(WORKDIR, exist_ok=True)

# ---- 2) Upload files or a zipped folder ----
print("Upload DOOR files (or a .zip containing them):")
uploaded = files.upload()

# Move uploads into staging and unzip any zips
for fname in uploaded.keys():
    src = f"/content/{fname}"
    dst = os.path.join(UPLOAD_STAGING, fname)
    shutil.move(src, dst)

    if fname.lower().endswith(".zip"):
        with zipfile.ZipFile(dst, "r") as z:
            z.extractall(UPLOAD_STAGING)
        os.remove(dst)

# ---- 3) Collect candidate file paths (recursively) ----
# Adjust extensions if you have other file types
r = []
for ext in ("*.csv", "*.tsv", "*.txt"):
    r.extend(glob.glob(os.path.join(UPLOAD_STAGING, "**", ext), recursive=True))

print(f"Found {len(r)} candidate files in uploads.")

# ---- 4) Working-file selection logic (fixed) ----
workingfiles = []

door_pattern = re.compile(r"^DOOR\d{3}_\d{6}_\d{2,3}\.(csv|txt|tsv)$", re.IGNORECASE)

skipped_non_door = 0
skipped_bad = 0

for f in r:
    fname = os.path.basename(f)

    # Skip non-DOOR files (e.g., metadata file)
    if not door_pattern.match(fname):
        print(f"SKIP (not DOOR file): {fname}")
        skipped_non_door += 1
        continue

    # Read and validate
    try:
        df = pd.read_csv(f)
    except Exception as e:
        print(f"SKIP (cannot read): {fname} | {e}")
        skipped_bad += 1
        continue

    # Basic column check (avoid KeyError)
    if ("Door" not in df.columns) or ("Event" not in df.columns):
        print(f"SKIP (missing Door/Event columns): {fname}")
        skipped_bad += 1
        continue

    if len(df) == 0:
        print(f"{fname} removed (empty)")
        os.remove(f)
        continue

    # Your logic
    if 1 not in pd.Series(df["Door"]).dropna().unique():
        print(f"{fname} not empty but nothing happened - SKIP")
        continue

    event_n = int(np.nanmax(pd.to_numeric(df["Event"], errors="coerce")))
    print(f"{fname} this file seems work, event#: {event_n}")

    # Copy into WORKDIR
    new_file_path = os.path.join(WORKDIR, fname)
    shutil.copy(f, new_file_path)
    print(f"Copied: {f} -> {new_file_path}")

    workingfiles.append(new_file_path)

print("\nSummary:")
print(f"Working files copied: {len(workingfiles)}")
print(f"Skipped non-DOOR files: {skipped_non_door}")
print(f"Skipped unreadable/malformed files: {skipped_bad}")

print("\nWorking files:")
print(workingfiles)



In [ ]:
# @title Upload a file containing Test mouse information. Please read the instructions below.
from google.colab import files
import pandas as pd

print("""
Please upload a file containing **Test mouse information data**.

Required columns (case-sensitive):
- starttime
- mode
- device_number
- sex

Optional columns:
- test_id
- genotype
- treatment
- treat_starttime

Column definitions:
- starttime: Date and time when the test started
- mode: Experimental condition. Examples include:
    - "Empty Cage"
    - "Social Stimulus"
    - "Object Stimulus"
    - or a specific individual / stimulus name
- device_number: Numeric device identifier
- test_id: Animal or test identifier (optional)
- sex: Biological sex (e.g., m, f)
- genotype: Genotype label (optional)
- treatment: Drug or treatment condition (optional)
- treat_starttime: Date and time when the treatment started (optional). The data within the 24 h after the treatment start time will be assigned with the treatment condition by default.

The uploaded file must have an extension that can be read into a pandas DataFrame
(e.g., .csv, .tsv, .xlsx). This information will be merged with the dataset later.
""")

# ---- Example shown as a DataFrame for perfect alignment ----
example_df = pd.DataFrame({
    "starttime": [
        "7/7/2025 14:55",
        "7/7/2025 15:11",
        "7/7/2025 15:01",
        "7/16/2025 15:09"
    ],
    "mode": [
        "Social Stimulus",
        "Empty Cage",
        "Empty Cage",
        "Object Stimulus"
    ],
    "device_number": [0, 1, 2, 2],
    "test_id": ["125-25", "", "78-67", "78-67"],
    "sex": ["m", "m", "m", "m"],
    "genotype": ["wt", "het", "ko", "ko"],
    "treatment": ["saline", "", "", "oxytocin"],
    "treat_starttime": ["7/7/2025 16:25", "", "", "7/16/2025 16:29"]
})

print("Example format:")
display(example_df)
print('\n')

# ---- Upload ----
UPLOAD_STAGING = "/content/upload_staging"
os.makedirs(UPLOAD_STAGING, exist_ok=True)

CANONICAL_NAME = "test_mouse_info.csv"
META_PATH = os.path.join(UPLOAD_STAGING, CANONICAL_NAME)

print("Upload the test mouse info file (csv / tsv / xlsx).",'\n')
print("You may cancel to reuse the existing file if one is already present.",'\n')

uploaded = files.upload()

# ---- Case 1: user uploaded a file ----
if len(uploaded) == 1:
    uploaded_name = list(uploaded.keys())[0]
    src = f"/content/{uploaded_name}"

    # Overwrite existing canonical file if present
    if os.path.exists(META_PATH):
        os.remove(META_PATH)

    shutil.move(src, META_PATH)
    print(f"Test mouse info file saved as: {META_PATH}")

# ---- Case 2: user canceled upload ----
elif len(uploaded) == 0:
    if os.path.exists(META_PATH):
        print(f"No file uploaded. Using existing test mouse info: {META_PATH}")
    else:
        raise FileNotFoundError(
            "No test mouse info file uploaded and no existing test_mouse_info.csv found."
        )

# ---- Case 3: user uploaded more than one file ----
else:
    raise ValueError("Please upload exactly ONE test mouse info file.")

# ---- Preview ----
meta = pd.read_csv(META_PATH)
print("Preview of test mouse info:")
display(meta)



In [ ]:
# @title This code will concate all individuals and merge the Test mouse information into your raw data
# @markdown Generated files are available to download in the next cell.
# =========================
# FULL PIPELINE (Colab) - uses DOOR columns: Datetime, Device_Number
# Merge key: Device_Number (DOOR) <-> device_number (metadata)
# Optional metadata cols are truly optional
# =========================

import os, re, glob
import pandas as pd
import numpy as np

# -------------------------
# 0) PATHS (EDIT THESE)
# -------------------------
DOOR_ROOT = "/content/upload_staging"
META_PATH = "/content/upload_staging/test_mouse_info.csv"
WORKDIR = "/content/working_files"
OUTPUT_ROOT = "/content/door_outputs"
os.makedirs(WORKDIR, exist_ok=True)

# -------------------------
# 1) READ TEST MOUSE INFO
# -------------------------
def read_table_any(path: str) -> pd.DataFrame:
    ext = os.path.splitext(path)[1].lower()
    if ext == ".csv":
        return pd.read_csv(path)
    if ext in [".tsv", ".txt"]:
        return pd.read_csv(path, sep="\t")
    if ext in [".xlsx", ".xls"]:
        return pd.read_excel(path)
    raise ValueError(f"Unsupported metadata file extension: {ext}")

meta = read_table_any(META_PATH)

required = ["starttime", "mode", "device_number", "sex"]
missing = [c for c in required if c not in meta.columns]
if missing:
    raise ValueError(f"Metadata file missing required columns: {missing}")

# Parse required fields
meta["starttime"] = pd.to_datetime(meta["starttime"], errors="coerce")
if meta["starttime"].isna().any():
    bad = meta[meta["starttime"].isna()]
    raise ValueError(f"Could not parse some starttime values:\n{bad}")

meta["device_number"] = pd.to_numeric(meta["device_number"], errors="raise").astype(int)
meta["mode"] = meta["mode"].astype(str)
meta["sex"] = meta["sex"].astype(str)

# Optional columns: only if present
OPTIONAL_COLS = ["test_id", "genotype", "treatment", "treat_starttime"]
present_optional = [c for c in OPTIONAL_COLS if c in meta.columns]
if "treat_starttime" in meta.columns:
    meta["treat_starttime"] = pd.to_datetime(meta["treat_starttime"], errors="coerce")

print("Optional columns present in metadata:", present_optional)

# -------------------------
# 2) FIND DOOR FILES
# -------------------------
# Keep strict DOOR naming (optional). If you want to include all csvs, remove this check.
door_pat = re.compile(r"^DOOR\d{3}_\d{6}_\d{2}\.(csv|tsv|txt)$", re.IGNORECASE)

door_files = []
for ext in ("*.csv", "*.tsv", "*.txt"):
    for f in glob.glob(os.path.join(DOOR_ROOT, "**", ext), recursive=True):
        if door_pat.match(os.path.basename(f)):
            door_files.append(f)

door_files = sorted(door_files)
print(f"Found {len(door_files)} DOOR files.")

# -------------------------
# 3) HELPERS: read DOOR key + file starttime from Datetime
# -------------------------
def read_door_device_number_and_starttime(filepath: str):
    df = pd.read_csv(filepath)

    if "Device_Number" not in df.columns:
        raise ValueError(f"Missing Device_Number in {os.path.basename(filepath)}")

    dev = pd.to_numeric(df["Device_Number"], errors="coerce").dropna().unique()
    if len(dev) == 0:
        raise ValueError(f"Device_Number is empty/unparseable in {os.path.basename(filepath)}")
    if len(dev) > 1:
        # Not fatal; warn and take the first
        print(f"WARNING: multiple Device_Number values in {os.path.basename(filepath)} -> {dev}. Using {int(dev[0])}.")

    device_number = int(dev[0])

    if "Datetime" in df.columns:
        t = pd.to_datetime(df["Datetime"], errors="coerce", format = 'mixed').dropna()
        if len(t) > 0:
            return device_number, t.min()

    # fallback: file modified time
    t0 = pd.to_datetime(os.path.getmtime(filepath), unit="s", origin="unix")
    return device_number, t0

# -------------------------
# 4) BUILD MODE WINDOWS PER device_number (from metadata)
# -------------------------
meta_sorted = meta.sort_values(["device_number", "starttime"]).copy()

def build_mode_windows(df_group: pd.DataFrame):
    rows = df_group.sort_values("starttime").to_dict("records")
    windows = []
    for i, row in enumerate(rows):
        start = row["starttime"]
        end = rows[i+1]["starttime"] if i+1 < len(rows) else pd.Timestamp.max

        payload = {
            "starttime": row["starttime"],
            "mode": row["mode"],
            "sex": row["sex"],
        }
        if "test_id" in meta.columns:
            payload["test_id"] = row.get("test_id", pd.NA)
        if "genotype" in meta.columns:
            payload["genotype"] = row.get("genotype", pd.NA)

        windows.append({"start": start, "end": end, "payload": payload})
    return windows

mode_windows_by_device = {
    dev: build_mode_windows(g)
    for dev, g in meta_sorted.groupby("device_number", dropna=False)
}
print(f"Built mode schedules for {len(mode_windows_by_device)} device_number(s).")

# -------------------------
# 5) TREATMENT WINDOWS (ONLY IF BOTH COLS EXIST)
# -------------------------
HAS_TREAT_WINDOWS = ("treat_starttime" in meta.columns) and ("treatment" in meta.columns)

def build_treatment_windows(df_group: pd.DataFrame):
    if not HAS_TREAT_WINDOWS:
        return []
    g = df_group.copy()
    g = g[g["treat_starttime"].notna()]
    tw = []
    for _, row in g.iterrows():
        if pd.isna(row["treatment"]):
            continue
        start = row["treat_starttime"]
        end = start + pd.Timedelta(hours=24)
        tw.append({"start": start, "end": end, "treatment": row["treatment"]})
    tw.sort(key=lambda d: d["start"])
    return tw

treat_windows_by_device = {}
if HAS_TREAT_WINDOWS:
    treat_windows_by_device = {
        dev: build_treatment_windows(g)
        for dev, g in meta_sorted.groupby("device_number", dropna=False)
    }

def apply_treatment_windows(df: pd.DataFrame, twindows: list) -> pd.DataFrame:
    """
    If treatment windows exist, assign treatment ONLY within 24h after treat_starttime.
    Outside windows, treatment is blank/NA.
    """
    if not HAS_TREAT_WINDOWS:
        return df

    out = df.copy()
    if "Datetime" not in out.columns or len(twindows) == 0:
        out["treatment"] = pd.NA
        return out

    dt = pd.to_datetime(out["Datetime"], errors="coerce")
    out["_dt"] = dt
    out["treatment"] = pd.NA

    for w in twindows:
        mask = (out["_dt"] >= w["start"]) & (out["_dt"] < w["end"])
        out.loc[mask, "treatment"] = w["treatment"]

    out.drop(columns=["_dt"], inplace=True)
    return out

# -------------------------
# 6) ENRICH DOOR FILES + BUILD DICT FOR CONCAT LOOP
# -------------------------
def clean_mode(s: str) -> str:
    return re.sub(r"\s+", "", str(s))

files_by_key = {}  # device_number -> mode_clean -> list of enriched DataFrames

for f in door_files:
    try:
        device_number, file_t0 = read_door_device_number_and_starttime(f)
    except Exception as e:
        print(f"SKIP (could not read key/time): {os.path.basename(f)} | {e}")
        continue

    if device_number not in mode_windows_by_device:
        print(f"SKIP (no metadata schedule for device_number={device_number}): {os.path.basename(f)}")
        continue

    windows = mode_windows_by_device[device_number]

    # choose mode window
    if len(windows) == 1:
        chosen = windows[0]
    else:
        chosen = None
        for w in windows:
            if (file_t0 >= w["start"]) and (file_t0 < w["end"]):
                chosen = w
                break
        if chosen is None:
            prev = [w for w in windows if file_t0 >= w["start"]]
            chosen = prev[-1] if prev else windows[0]

    payload = chosen["payload"]
    mode_clean = clean_mode(payload["mode"])

    df = pd.read_csv(f)

    # merge required fields
    df["starttime"] = payload["starttime"]
    df["mode"] = payload["mode"]
    df["sex"] = payload["sex"]

    # optional fields
    if "test_id" in payload:
        df["test_id"] = payload["test_id"]
    if "genotype" in payload:
        df["genotype"] = payload["genotype"]

    # treatment windows (only if enabled)
    if HAS_TREAT_WINDOWS:
        df = apply_treatment_windows(df, treat_windows_by_device.get(device_number, []))

    # store dataframe directly (NO saving)
    files_by_key.setdefault(device_number, {}).setdefault(mode_clean, []).append(df)

print("Enrichment complete (in memory only).")

if len(files_by_key) > 0:
    d0 = sorted(files_by_key.keys())[0]
    print("Example mapping:", d0, {k: len(v) for k, v in files_by_key[d0].items()})

# -------------------------
# 7) CONCAT LOOP (YOUR STRUCTURE)
# -------------------------
# =========================
# FINAL OUTPUTS + PER-MODE GLOBAL OUTPUTS
# Assumes you already built: files_by_key (device_number -> mode_clean -> list of DataFrames)
# and you have: remove_button_event(df) defined.
# Writes:
#   - DOORxxx_mode.csv (per device per mode)
#   - all_DOORs.csv (all devices/all modes)
#   - all_DOORs_mode.csv (global per mode)
# =========================

def remove_button_event(df, baseline = 0):
    pairs_to_remove = df[df['Door'] == -2][['Device_Number', 'Event']].drop_duplicates()
    for index, row in pairs_to_remove.iterrows():
        df = df[~((df['Device_Number'] == row['Device_Number']) & (df['Event'] == row['Event']))]
    df1 = df[df["Door"] >= baseline].reset_index(drop=True)
    return df1

WORKDIR = "/content/working_files"
os.makedirs(WORKDIR, exist_ok=True)

alldoors = pd.DataFrame()
alldoors_by_mode = {}  # mode_clean -> DataFrame

for device_number, modes_dict in files_by_key.items():
    for mode_clean, df_list in modes_dict.items():
        comb = pd.DataFrame()
        last_trial = 0

        # sort files for this (device, mode) by earliest Datetime in each file
        df_list = sorted(
            df_list,
            key=lambda d: pd.to_datetime(d["Datetime"], errors="coerce").min()
        )

        for df in df_list:
            df1 = remove_button_event(df)
            df1 = df1.query("Event > 0").copy()

            df1.loc[:, "Event"] += last_trial
            last_trial = df1.loc[len(df1) - 1, "Event"]

            comb = pd.concat([comb, df1], ignore_index=True)


        # ---- Accumulate global (all modes) ----
        alldoors = pd.concat([alldoors, comb], ignore_index=True)

        # ---- Accumulate global per-mode ----
        if mode_clean not in alldoors_by_mode:
            alldoors_by_mode[mode_clean] = comb.copy()
        else:
            alldoors_by_mode[mode_clean] = pd.concat(
                [alldoors_by_mode[mode_clean], comb],
                ignore_index=True
            )

        # ---- Save per-device per-mode ----
        comb_path = os.path.join(OUTPUT_ROOT, f"DOOR{device_number:03d}_{mode_clean}.csv")
        comb.to_csv(comb_path, index=False)

# ---- Save global all modes ----
all_path = os.path.join(OUTPUT_ROOT, "all_DOORs.csv")
alldoors.to_csv(all_path, index=False)

# ---- Save global per-mode ----
for mode_clean, df_mode in alldoors_by_mode.items():
    mode_path = os.path.join(OUTPUT_ROOT, f"all_DOORs_{mode_clean}.csv")
    df_mode.to_csv(mode_path, index=False)

print("Saved outputs to:", OUTPUT_ROOT)
print("- all_DOORs.csv")
print("- all_DOORs_<mode>.csv for each mode")
print("- DOORxxx_<mode>.csv for each device/mode")


In [ ]:
# @title Download merged data as a zip file
import shutil
from google.colab import files

zip_path = shutil.make_archive("/content/door_outputs", "zip", OUTPUT_ROOT)
# files.download(zip_path)


In [ ]:
# @title Choose whether to upload information for Stimulus mice.
from google.colab import files
import os, shutil
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, clear_output

UPLOAD_STAGING = "/content/upload_staging"
os.makedirs(UPLOAD_STAGING, exist_ok=True)

CANONICAL_STIM_NAME = "stimulus_mouse_info.csv"
STIM_PATH = os.path.join(UPLOAD_STAGING, CANONICAL_STIM_NAME)

def read_table_any(path: str) -> pd.DataFrame:
    ext = os.path.splitext(path)[1].lower()
    if ext == ".csv":
        return pd.read_csv(path)
    if ext in [".tsv", ".txt"]:
        return pd.read_csv(path, sep="\t")
    if ext in [".xlsx", ".xls"]:
        return pd.read_excel(path)
    raise ValueError(f"Unsupported file extension: {ext}")

print("""
Stimulus mouse info is OPTIONAL.
If you choose to include it, required columns are (case-sensitive):
- datetime
- device_number
- stimulus_id
""")

example_stimulus_df = pd.DataFrame({
    "datetime": [
        "10/16/2024 15:16","10/16/2024 15:16","10/16/2024 15:16",
        "10/19/2024 9:34","10/19/2024 9:34","10/19/2024 9:34",
        "10/22/2024 14:30","10/22/2024 14:30","10/22/2024 14:30",
    ],
    "device_number": [0,1,2,0,1,2,0,1,2],
    "stimulus_id": ["29169-1","29169-2","29169-3","29169-4","29169-1","29169-2","29169-3","29169-4","29169-1"]
})
print("Example format:")
display(example_stimulus_df)

# ---- UI ----
choice = widgets.RadioButtons(
    options=["No (skip stimulus info)", "Yes (upload stimulus info)"],
    description="Stimulus info:",
    disabled=False
)

confirm = widgets.Button(description="Continue", button_style="primary")

out = widgets.Output()

def on_confirm_clicked(b):
    with out:
        clear_output()
        if choice.value.startswith("No"):
            print("Stimulus mouse info will NOT be used.")
            globals()["stimulus_info"] = None
        else:
            print("Upload stimulus mouse info (csv / tsv / xlsx):")
            uploaded = files.upload()

            if len(uploaded) != 1:
                raise ValueError("Please upload exactly ONE stimulus mouse info file.")

            uploaded_name = list(uploaded.keys())[0]
            src = f"/content/{uploaded_name}"

            if os.path.exists(STIM_PATH):
                os.remove(STIM_PATH)
            shutil.move(src, STIM_PATH)

            stim = read_table_any(STIM_PATH)

            needed = {"datetime", "device_number", "stimulus_id"}
            if not needed.issubset(set(stim.columns)):
                raise ValueError(f"Stimulus file missing required columns {needed}.")

            stim["device_number"] = pd.to_numeric(stim["device_number"], errors="coerce")
            stim["datetime"] = pd.to_datetime(stim["datetime"], errors="coerce", format="mixed")
            stim = stim.dropna(subset=["device_number", "datetime"]).copy()
            stim["device_number"] = stim["device_number"].astype(int)
            stim = stim.sort_values(["device_number", "datetime"])

            globals()["stimulus_info"] = stim
            print("Stimulus mouse info loaded:")
            display(stim)

display(choice, confirm, out)
confirm.on_click(on_confirm_clicked)



In [ ]:
# @title This code classifies behavior and interaction types and generates summarized data file(s) available to download.
import numpy as np
import pandas as pd
from google.colab import files
import os
import warnings
warnings.filterwarnings("ignore")

# -------------------------
# Helper functions (same as yours, kept)
# -------------------------
def first_value_below_threshold(prox_values, threshold=50):
    below_threshold = np.where(prox_values < threshold)[0]
    return below_threshold[0] if below_threshold.size > 0 else -1

def last_value_below_threshold(prox_values, threshold=50):
    below_threshold = np.where(prox_values < threshold)[0]
    return below_threshold[-1] if below_threshold.size > 0 else -1

def compute_wait_time(row):
    a1 = row.get('firstapproach1', np.nan)
    a2 = row.get('firstapproach2', np.nan)
    l2 = row.get('leavetime2', np.nan)

    if pd.notna(a1) and pd.notna(a2):
        return a1 - a2
    elif pd.isna(a1) and pd.notna(a2) and pd.notna(l2):
        return l2 - a2
    else:
        return np.nan

def assign_stimulus_id(all_doors_df, stimulus_info_df):
    # expects: all_doors_df has ['device_number','datetime'], stimulus_info_df has ['device_number','datetime','stimulus_id']
    out = all_doors_df.copy()
    out["stimulus_id"] = pd.NA

    # ensure sorted for forward-fill logic
    out = out.sort_values(["device_number", "datetime"])

    for device in stimulus_info_df["device_number"].dropna().unique():
        device = int(device)
        stim = stimulus_info_df[stimulus_info_df["device_number"] == device].sort_values("datetime")
        if stim.empty:
            continue

        dmask = out["device_number"] == device
        if not dmask.any():
            continue

        # for each stim change, set stimulus_id for rows at/after change
        for _, stim_row in stim.iterrows():
            cond = dmask & (out["datetime"] >= stim_row["datetime"])
            out.loc[cond, "stimulus_id"] = stim_row["stimulus_id"]

    return out

# -------------------------
# Colab: choose which all_DOORs files to process
# -------------------------
OUTPUT_ROOT = "/content/door_outputs"  # where you saved all_DOORs*.csv earlier

def list_all_doors_candidates(output_root=OUTPUT_ROOT):
    if not os.path.exists(output_root):
        raise FileNotFoundError(f"{output_root} not found. Did you generate outputs yet?")
    candidates = sorted([
        os.path.join(output_root, f)
        for f in os.listdir(output_root)
        if f.lower().endswith(".csv") and f.startswith("all_DOORs")
    ])
    return candidates

candidates = list_all_doors_candidates()
print("Found these all_DOORs candidate files:")
for i, p in enumerate(candidates):
    print(f"[{i}] {os.path.basename(p)}")

choice = input("Enter indices to process (comma-separated), or 'all': ").strip().lower()
if choice == "all":
    doorfiles_to_process = candidates
else:
    idx = [int(x.strip()) for x in choice.split(",") if x.strip() != ""]
    doorfiles_to_process = [candidates[i] for i in idx]

print("\nWill process:")
for p in doorfiles_to_process:
    print(" -", os.path.basename(p))

# -------------------------
# Core analysis function: takes a DataFrame (already loaded), not a filename
# -------------------------
def approach_time_analysis_df(data: pd.DataFrame, stimulus_info: pd.DataFrame | None = None) -> pd.DataFrame:
    data = data.copy()

    # Normalize column names expected by your code:
    # - Your DOOR exports use Device_Number and Datetime (based on your earlier pipeline)
    # - Your event_df uses lowercase device_number / datetime
    if "Device_Number" in data.columns and "device_number" not in data.columns:
        data["device_number"] = pd.to_numeric(data["Device_Number"], errors="coerce")
    if "Datetime" in data.columns and "datetime" not in data.columns:
        data["datetime"] = pd.to_datetime(data["Datetime"], errors="coerce", format="mixed")

    # Make sure these exist
    needed_cols = ["Device_Number", "Datetime", "Event", "Seconds", "Door", "Prox1", "Prox2", "mode"]
    missing = [c for c in needed_cols if c not in data.columns]
    if missing:
        raise ValueError(f"Input all_DOORs file is missing required columns: {missing}")

    # Apply Prox caps
    data["Prox1"] = data["Prox1"].apply(lambda x: min(x, 100))
    data["Prox2"] = data["Prox2"].apply(lambda x: min(x, 100))

    # Identify event starts
    data["Event_Start"] = data["Event"].diff().ne(0)
    event_starts = data[data["Event_Start"]]

    # Start indices per device + mode
    event_indices = (
        event_starts.groupby(["Device_Number", "mode"])
        .apply(lambda x: x.index.tolist())
        .reset_index(name="Start_Indices")
    )

    event_data = []

    def process_event(device_number, index_list, mode):
        for index in index_list:
            event_slice = data[
                (data["Device_Number"] == device_number) &
                (data["Event"] == data.loc[index, "Event"])
            ].sort_values("Seconds")

            door_rows = event_slice.loc[event_slice["Door"] == 1]
            if door_rows.empty or len(door_rows) < 4:
                continue

            s = pd.to_numeric(event_slice["Door"], errors="coerce")
            d = s.diff()

            def first_change_idx(diff_series, delta):
                ix = diff_series.index[diff_series.eq(delta)]
                return ix.min() if len(ix) else None

            dooropen_idx  = first_change_idx(d,  1)
            doorclose_idx = first_change_idx(d, -1)

            if (dooropen_idx is not None) and (doorclose_idx is not None):
                if dooropen_idx < doorclose_idx:
                    open_idx, close_idx, status = dooropen_idx, doorclose_idx, "normal"
                else:
                    open_idx, close_idx, status = None, doorclose_idx, "open start"
            elif dooropen_idx is not None:
                open_idx, close_idx, status = dooropen_idx, None, "close fail"
            elif doorclose_idx is not None:
                open_idx, close_idx, status = None, doorclose_idx, "open start"
            else:
                open_idx, close_idx, status = None, None, "open start close fail"

            if open_idx is not None:
                event_time = event_slice.loc[open_idx, "Seconds"]
            else:
                event_time = door_rows["Seconds"].iloc[0]

            if close_idx is not None:
                last_event_time = event_slice.loc[close_idx, "Seconds"] - event_time
            else:
                last_event_time = door_rows["Seconds"].iloc[-1] - event_time

            group = event_slice.reset_index()
            adjusted_seconds = group["Seconds"] - event_time
            start_time = adjusted_seconds.iloc[0]
            end_time = min(last_event_time, adjusted_seconds.iloc[-1])

            time_window = np.round(np.arange(start_time, end_time, 0.1), 2)

            if len(group) > 1:
                prox1_values = np.round(np.interp(time_window, adjusted_seconds, group["Prox1"]), 1)
                prox2_values = np.round(np.interp(time_window, adjusted_seconds, group["Prox2"]), 1)

                first_index1 = first_value_below_threshold(prox1_values)
                first_index2 = first_value_below_threshold(prox2_values)
                leave_index1 = last_value_below_threshold(prox1_values)
                leave_index2 = last_value_below_threshold(prox2_values)

                first_approach2 = time_window[first_index2] if first_index2 != -1 else 99
                first_approach1 = time_window[first_index1] if first_index1 != -1 else 99
                last_approach2  = time_window[leave_index2] if leave_index2 != -1 else 99
                last_approach1  = time_window[leave_index1] if leave_index1 != -1 else 99

                doortime1after0 = np.sum(prox1_values[np.where(time_window >= 0)] <= 50) / 10
                totaldoortime1  = np.sum(prox1_values <= 50) / 10
                doortime2after0 = np.sum(prox2_values[np.where(time_window >= 0)] <= 50) / 10
                totaldoortime2  = np.sum(prox2_values <= 50) / 10

                approach_prox1 = np.where(prox1_values <= 50)[0]
                if len(np.where(time_window >= 0)[0]) > 0:
                    approach_prox1 = approach_prox1[approach_prox1 >= np.where(time_window >= 0)[0][0]]

                approach_prox2 = np.where(prox2_values <= 50)[0]
                if len(np.where(time_window >= 0)[0]) > 0:
                    approach_prox2 = approach_prox2[approach_prox2 >= np.where(time_window >= 0)[0][0]]

                recisum1 = np.sum(prox2_values[approach_prox1] <= 50) / 10 if len(approach_prox1) else 0
                recisum2 = np.sum(prox1_values[approach_prox2] <= 50) / 10 if len(approach_prox2) else 0

                if first_approach2 < 99:
                    behavior = "Seeking"
                    itype = "Reciprocal interaction" if recisum2 > 1 else "Nonreciprocal interaction"
                else:
                    behavior = "Exploring"
                    itype = "Late interaction" if recisum2 > 1 else "No interaction"

                # Pull optional metadata columns if present in the all_DOORs file
                extra = {}
                for col in ["sex", "genotype", "test_id", "treatment"]:
                    if col in data.columns:
                        extra[col] = data.loc[index, col]

                event_data.append({
                    "datetime": data.loc[index, "Datetime"],
                    "device_number": device_number,
                    "mode": mode,
                    "event": data.loc[index, "Event"],
                    "time_window": time_window,
                    "door_status": status,
                    "prox1_values": prox1_values,
                    "prox2_values": prox2_values,
                    "behavior": behavior,
                    "interaction": itype,
                    "firstapproach1": first_approach1 if first_approach1 != 99 else np.nan,
                    "leavetime1": last_approach1 if last_approach1 != 99 else np.nan,
                    "firstapproach2": first_approach2 if first_approach2 != 99 else np.nan,
                    "leavetime2": last_approach2 if last_approach2 != 99 else np.nan,
                    "stim_interactiontime": recisum1,
                    "test_interactiontime": recisum2,
                    "totalappduration1": totaldoortime1,
                    "totalappduration2": totaldoortime2,
                    "appduration1after0": doortime1after0,
                    "appduration2after0": doortime2after0,
                    "approach_latency2": first_approach2 - start_time if first_approach2 != 99 else np.nan,
                    "approach_latency1": first_approach1 - start_time if first_approach1 != 99 else np.nan,
                    "door_operating_time": 0 - start_time,
                    **extra
                })

    for _, row in event_indices.iterrows():
        process_event(row["Device_Number"], row["Start_Indices"], row["mode"])

    event_df = pd.DataFrame(event_data)
    if event_df.empty:
        return event_df

    event_df["wait_time"] = event_df.apply(compute_wait_time, axis=1)

    # Standardize datetime for merging/stimulus assignment
    event_df["datetime"] = pd.to_datetime(event_df["datetime"], errors="coerce", format="mixed")

    # Optional stimulus assignment
    if stimulus_info is not None:
        event_df = assign_stimulus_id(event_df, stimulus_info)

    return event_df

# -------------------------
# Run on selected all_DOORs files and concatenate results
# -------------------------
event_dfs = []
for doorfile in doorfiles_to_process:
    print("Processing:", os.path.basename(doorfile))
    data = pd.read_csv(doorfile)
    ev = approach_time_analysis_df(data, stimulus_info=stimulus_info)
    ev["source_file"] = os.path.basename(doorfile)
    event_dfs.append(ev)

event_df_all = pd.concat(event_dfs, ignore_index=True) if len(event_dfs) else pd.DataFrame()
print("Done. Rows:", len(event_df_all))
display(event_df_all)


In [ ]:
# @title Download the file(s) if you would like to.
from google.colab import files
import os

OUTPUT_ROOT = "/content/door_outputs"
os.makedirs(OUTPUT_ROOT, exist_ok=True)

out_path = os.path.join(OUTPUT_ROOT, "event_df_all.csv")
event_df_all.to_csv(out_path, index=False)

print("Saved:", out_path)

# Download to the user's computer
files.download(out_path)

# also save per-source file outputs
for ev in event_dfs:
    src = ev["source_file"].iloc[0] if len(ev) else "unknown"
    base = os.path.splitext(src)[0]
    p = os.path.join(OUTPUT_ROOT, f"event_{base}.csv")
    ev.to_csv(p, index=False)
    print("Saved:", p)

In [1]:
# @title Definitions of data column names
# Data dictionary for the event-level output table
event_table_dictionary = pd.DataFrame([
    {"Key": "datetime",
     "Explanation": "Timestamp (date and time) when the event occurs."},

    {"Key": "device_number",
     "Explanation": "Door device identifier (the device number configured on the door system)."},

    {"Key": "mode",
     "Explanation": (
         "Experimental condition during the event. Examples: "
         "'Empty Cage', 'Social Stimulus', 'Object Stimulus', or a specific individual/stimulus name."
     )},

    {"Key": "event",
     "Explanation": "Event index (identifier) assigned to a single door event."},

    {"Key": "time_window",
     "Explanation": (
         "Event-aligned time vector (seconds) spanning from nose poke through door closing, "
         "sampled every 0.1 s."
     )},

    {"Key": "door_status",
     "Explanation": (
         "Door transition classification for the event (e.g., normal, close fail, open start), "
         "based on detected door open/close transitions."
     )},

    {"Key": "prox1_values",
     "Explanation": "Left / stimulus-side proximity trace across time_window (capped at 100; lower = closer)."},

    {"Key": "prox2_values",
     "Explanation": "Right / test-mouse-side proximity trace across time_window (capped at 100; lower = closer)."},

    {"Key": "behavior",
     "Explanation": "Test mouse behavior label for the event: 'Seeking' or 'Exploring'."},

    {"Key": "interaction",
     "Explanation": "Interaction category for the event: Reciprocal interaction, Nonreciprocal interaction, Late interaction, or No interaction."},

    {"Key": "firstapproach1",
     "Explanation": "First time (s) the left/stimulus-side proximity crosses the approach threshold (≤ 5 cm). NaN if no approach."},

    {"Key": "leavetime1",
     "Explanation": "Last time (s) the left/stimulus-side proximity is within approach threshold (≤ 5 cm). NaN if no approach."},

    {"Key": "firstapproach2",
     "Explanation": "First time (s) the right/test-mouse-side proximity crosses the approach threshold (≤ 5 cm). NaN if no approach."},

    {"Key": "leavetime2",
     "Explanation": "Last time (s) the right/test-mouse-side proximity is within approach threshold (≤ 5 cm). NaN if no approach."},

    {"Key": "stim_interactiontime",
     "Explanation": "Stimulus-side interaction duration (s) during the event."},

    {"Key": "test_interactiontime",
     "Explanation": "Test-mouse-side interaction duration (s) during the event."},

    {"Key": "totalappduration1",
     "Explanation": "Total duration (s) the left/stimulus-side is within approach threshold across the full event window."},

    {"Key": "totalappduration2",
     "Explanation": "Total duration (s) the right/test-mouse-side is within approach threshold across the full event window."},

    {"Key": "appduration1after0",
     "Explanation": "Left/stimulus-side approach duration (s) after time 0 (door fully open), within approach threshold."},

    {"Key": "appduration2after0",
     "Explanation": "Right/test-mouse-side approach duration (s) after time 0 (door fully open), within approach threshold."},

    {"Key": "approach_latency2",
     "Explanation": "Latency (s) for the right/test mouse to first approach after the event reference time. NaN if no approach."},

    {"Key": "approach_latency1",
     "Explanation": "Latency (s) for the left/stimulus side to first approach after the event reference time. NaN if no approach."},

    {"Key": "door_operating_time",
     "Explanation": "Door opening time (s): time from the start of the event window to when the door is fully open (time 0)."},
])

display(event_table_dictionary)


,Key,Explanation
0,datetime,Timestamp (date and time) when the event occurs.
1,device_number,Door device identifier (the device number conf...
2,mode,Experimental condition during the event. Examp...
3,event,Event index (identifier) assigned to a single ...
4,time_window,Event-aligned time vector (seconds) spanning f...
5,door_status,Door transition classification for the event (...
6,prox1_values,Left / stimulus-side proximity trace across ti...
7,prox2_values,Right / test-mouse-side proximity trace across...
8,behavior,Test mouse behavior label for the event: 'Seek...
9,interaction,Interaction category for the event: Reciprocal...
